# Formula 1 Race Finish Prediction using Linear Regression

In this notebook, we build a simple, interpretative predictive model using `scikit-learn` Linear Regression to predict a driver's final race finish position based on season indicators and driver factors.

## Goal:
Predict `positionOrder` (final finishing position, from 1 to 20+) using variables known *before* the race:
1. **`grid`**: The starting grid position (1 to 20+).
2. **`driver_experience`**: The number of career races the driver has participated in up to that point.
3. **`team_standing_points`**: The constructor's standing points prior to the race (as a proxy for car performance).
4. **`driver_form`**: The driver's average finishing position over their last 3 races.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = '../Data'

## 1. Load and Merge Data
We load races, results, drivers, constructors, and driver standings.

In [ ]:
def load_csv(name):
    return pd.read_csv(os.path.join(DATA_DIR, name)).replace(r'\\N', np.nan, regex=True).replace(r'\\N', np.nan)

results = load_csv('results.csv')
races = load_csv('races.csv')
drivers = load_csv('drivers.csv')
constructors = load_csv('constructors.csv')
driver_standings = load_csv('driver_standings.csv')

# Clean numeric fields
results['positionOrder'] = pd.to_numeric(results['positionOrder'])
results['grid'] = pd.to_numeric(results['grid'])
results['raceId'] = pd.to_numeric(results['raceId'])
results['driverId'] = pd.to_numeric(results['driverId'])
results['constructorId'] = pd.to_numeric(results['constructorId'])
races['raceId'] = pd.to_numeric(races['raceId'])
races['year'] = pd.to_numeric(races['year'])
races['round'] = pd.to_numeric(races['round'])

df = results.merge(races[['raceId', 'year', 'round', 'date']], on='raceId', how='left')
df = df.sort_values(by=['year', 'round', 'positionOrder'])
df.head()

## 2. Feature Engineering
Let's compute driver career experience (cumulative races driven) and the team's average finish or standing points.

In [ ]:
# 1. Driver Experience: cumulative count of races per driver
df['driver_experience'] = df.groupby('driverId').cumcount()

# 2. Driver Form: average positionOrder of the last 3 races (rolling average)
df['driver_form'] = df.groupby('driverId')['positionOrder'].shift(1).rolling(3, min_periods=1).mean()
# Fill NaNs (for drivers in their first 3 races) with their grid position
df['driver_form'] = df['driver_form'].fillna(df['grid'])

driver_standings['points'] = pd.to_numeric(driver_standings['points'])
driver_standings['raceId'] = pd.to_numeric(driver_standings['raceId'])
driver_standings['driverId'] = pd.to_numeric(driver_standings['driverId'])

driver_standings['prev_points'] = driver_standings.groupby('driverId')['points'].shift(1).fillna(0)

df = df.merge(driver_standings[['raceId', 'driverId', 'prev_points']], on=['raceId', 'driverId'], how='left')
df['prev_points'] = df['prev_points'].fillna(0)

# Inspect engineered features
df[['year', 'raceId', 'driverId', 'grid', 'driver_experience', 'driver_form', 'prev_points', 'positionOrder']].tail(10)

## 3. Data Split & Model Training
We will filter the dataset for races since 2010 to build our regression on modern data, and split into train/test sets.

In [ ]:
model_df = df[df['year'] >= 2010].dropna(subset=['positionOrder', 'grid', 'driver_experience', 'driver_form', 'prev_points'])

features = ['grid', 'driver_experience', 'driver_form', 'prev_points']
X = model_df[features]
y = model_df['positionOrder']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

## 4. Model Evaluation
Let's see how our Linear Regression model performs on the test set.

In [ ]:
y_pred = model.predict(X_test)

print(f"R2 Score (Variance explained): {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f} positions")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} positions")

## 5. Model Coefficients & Interpretation
Let's print the model coefficients to see what variables contribute most to predicting the finishing position.

In [ ]:
coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_
})
coefficients = coefficients.sort_values(by='Coefficient', ascending=False)

sns.barplot(data=coefficients, x='Coefficient', y='Feature', hue='Feature', legend=False, palette='coolwarm')
plt.title('Feature Coefficients in F1 Finish Regression')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.show()
print(coefficients)

### Interpretation:
- **`grid`** has a highly positive coefficient (around ~0.5). Every position further back on the grid shifts the expected finishing position about 0.5 positions back.
- **`prev_points`** (team points in standings) has a negative coefficient. More points in the standings (stronger team/driver) correlates with a lower (better) expected finishing position.
- **`driver_form`** also shows a positive coefficient, indicating that drivers who finished further back in recent races are expected to finish further back in the current race.

This simple model is easy to understand and can be trained/evaluated instantly inside our Streamlit dashboard.